# Create Carl-Zeiss-Stiftung Awards

Creates Carl-Zeiss-Stiftung (Carl Zeiss Foundation) awards from the foundation's
public project pages. ~226 projects funding AI, life sciences, and resource
efficiency research at German universities and research institutions.

**Prerequisites:**
- Run `scripts/local/carl_zeiss_to_s3.py` to scrape the project list and upload
  the parquet first.

**Data source:** carl-zeiss-stiftung.de project detail pages (HTML scrape, sitemap-driven)
**S3 location:** `s3a://openalex-ingest/awards/carl_zeiss/carl_zeiss_projects.parquet`

**Carl-Zeiss-Stiftung funder (OpenAlex):**
- funder_id: 4320309895
- ROR: https://ror.org/03ng4kg22
- DOI: 10.13039/100007569
- display_name: "Carl-Zeiss-Stiftung"

**Schema notes:**
- Currency is hardcoded to EUR by the download script (German foundation; all
  amounts on the site are in EUR; no per-record currency field exists).
- Amount is parsed from "Funding budget: 8.000.000 €" — German number formatting
  is normalised in the script.
- Period is parsed from "Period of time: January 2019 - December 2026". Day
  defaults to the 1st of the month.
- Funded institution → lead_investigator.affiliation.name (country = DE).


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.carl_zeiss_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/carl_zeiss/carl_zeiss_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.carl_zeiss_raw;

## Step 1.5: Inspect raw data

In [ ]:
%sql
DESCRIBE openalex.awards.carl_zeiss_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.carl_zeiss_raw LIMIT 5;

In [ ]:
%sql
-- Sanity-check amount/currency BEFORE transforming (per Step 1.6 in how-to)
SELECT
  COUNT(*) as rows,
  COUNT(amount_eur) as has_amount,
  ROUND(MIN(amount_eur), 0) as min_amount,
  ROUND(MAX(amount_eur), 0) as max_amount,
  ROUND(AVG(amount_eur), 0) as avg_amount,
  collect_set(currency) as currencies
FROM openalex.awards.carl_zeiss_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.carl_zeiss_awards
USING delta
AS
WITH czs_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320309895  -- Carl-Zeiss-Stiftung
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.slug)))) % 9000000000 as id,
    g.title as display_name,
    g.description,
    f.funder_id,
    g.slug as funder_award_id,
    TRY_CAST(g.amount_eur AS DOUBLE) as amount,
    g.currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    -- Funding type: rough mapping from Carl-Zeiss "Type of funding" field
    CASE
        WHEN LOWER(g.type_of_funding) LIKE '%individual%'
          OR LOWER(g.type_of_funding) LIKE '%fellowship%' THEN 'fellowship'
        WHEN LOWER(g.type_of_funding) LIKE '%project%' THEN 'research'
        ELSE 'grant'
    END as funding_type,
    -- Programme = funder_scheme (e.g. "CZS Nexus", "CZS Applied basic science")
    g.programme as funder_scheme,
    'carl_zeiss_stiftung' as provenance,
    TRY_TO_DATE(g.start_date, 'yyyy-MM-dd') as start_date,
    TRY_TO_DATE(g.end_date, 'yyyy-MM-dd') as end_date,
    YEAR(TRY_TO_DATE(g.start_date, 'yyyy-MM-dd')) as start_year,
    YEAR(TRY_TO_DATE(g.end_date, 'yyyy-MM-dd')) as end_year,
    struct(
        CAST(NULL AS STRING) as given_name,
        CAST(NULL AS STRING) as family_name,
        CAST(NULL AS STRING) as orcid,
        CAST(NULL AS DATE) as role_start,
        struct(
            g.funded_institution as name,
            'DE' as country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
        ) as affiliation
    ) as lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) as investigators,
    g.url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.slug)))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.carl_zeiss_raw g
CROSS JOIN czs_funder f
WHERE g.slug IS NOT NULL AND TRIM(g.slug) != '';

## Step 3: Insert into openalex_awards_raw at priority 38

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'carl_zeiss_stiftung' AND priority = 38;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    38 as priority
FROM openalex.awards.carl_zeiss_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) as total_czs_awards FROM openalex.awards.carl_zeiss_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amount,
    ROUND(MAX(amount), 0) AS max_amount,
    ROUND(AVG(amount), 0) AS avg_amount
FROM openalex.awards.carl_zeiss_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) as title, funder_scheme, amount, currency,
       start_date, end_date,
       lead_investigator.affiliation.name as institution
FROM openalex.awards.carl_zeiss_awards LIMIT 10;

In [ ]:
%sql
SELECT funding_type, COUNT(*) as cnt FROM openalex.awards.carl_zeiss_awards
GROUP BY funding_type ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_eur
FROM openalex.awards.carl_zeiss_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) FROM openalex.awards.carl_zeiss_awards GROUP BY funder.display_name;